# tensorcas — XGBoost large model analysis

Measures deduplication for large XGBoost boosters:
- **Baseline**: 1 000 rounds, 50 rounds/step (20 checkpoints) — warm-start boosting
- **DVC comparison**: honest binary (`.ubj`) vs tensorcas at large scale
- **Crossover analysis**: at what model size does tensorcas overtake DVC?

Key questions:
1. Does the `__skeleton__` tensor always change? Why?
2. At what round count does tensorcas's storage advantage emerge?
3. How does the binary UBJ format affect DVC's effective compression ratio?

In [ ]:
!pip install -q git+https://github.com/Olamyy/tensorcas.git@hash-cache-no-op-path zstandard xgboost

In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import xgboost as xgb

sys.path.insert(0, str(Path(".").resolve()))
from utils import (
    CHUNK_SIZE,
    extract_xgboost,
    measure_noop,
    measure_chunk_dedup,
    measure_crossrun,
    run_dvc_comparison,
    dvc_bytes,
    tensorcas_bytes,
    print_noop,
    print_chunk,
    print_crossrun,
    print_dvc_comparison,
    _fmt_bytes,
)

print("Imports OK")

## Configuration

In [ ]:
N_STEPS = 20
ROUNDS_PER_STEP = 50   # 1 000 rounds total
CHECKPOINT_DIR = Path("/tmp/tensorcas_xgboost_large")

print(f"Total rounds: {N_STEPS * ROUNDS_PER_STEP}")
print(f"Steps: {N_STEPS}, rounds/step: {ROUNDS_PER_STEP}")

## Training helpers

In [ ]:
def _make_dataset(seed: int = 0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((2000, 20)).astype(np.float32)
    y = (X[:, 0] + 0.5 * X[:, 1] > 0).astype(np.float32)
    return xgb.DMatrix(X, label=y)


def train_sequence(
    seed: int = 0,
    n_steps: int = N_STEPS,
    rounds_per_step: int = ROUNDS_PER_STEP,
    max_depth: int = 6,
    eta: float = 0.1,
):
    dtrain = _make_dataset(seed)
    params = {
        "max_depth": max_depth,
        "eta": eta,
        "objective": "binary:logistic",
        "seed": seed,
        "verbosity": 0,
    }
    boosters, booster = [], None
    for _ in range(n_steps):
        booster = xgb.train(
            params, dtrain,
            num_boost_round=rounds_per_step,
            xgb_model=booster,
            verbose_eval=False,
        )
        boosters.append(booster)
    return boosters


print("Training helpers OK")

## Baseline: 1 000-round warm-start run

In [ ]:
t0 = time.time()
print(f"Training {N_STEPS} steps × {ROUNDS_PER_STEP} rounds (seed=0)...", end=" ", flush=True)
baseline_boosters = train_sequence(seed=0)
print(f"{time.time() - t0:.1f}s")

In [ ]:
t0 = time.time()
print("Extracting tensors...", end=" ", flush=True)
baseline_seqs = [extract_xgboost(b) for b in baseline_boosters]
print(f"{time.time() - t0:.1f}s")

sample = baseline_seqs[-1]
n_tensors = len(sample)
total_bytes = sum(v.nbytes for v in sample.values())
print(f"Tensors per checkpoint (final): {n_tensors}")
print(f"Tensor bytes (final checkpoint): {_fmt_bytes(total_bytes)}")

### No-op fast path

Note: `__skeleton__` (the non-tree JSON metadata) changes every step as XGBoost
embeds round counts and model metadata in the header. Individual tree tensors
should be 100% stable once added.

In [ ]:
t0 = time.time()
noop_stats = measure_noop(baseline_seqs)
print_noop("xgboost large (baseline)", noop_stats, max_tensors=25)
print(f"\n  Measured in {time.time() - t0:.1f}s")

### Chunk-level reuse (changed tensors only)

In [ ]:
t0 = time.time()
chunk_stats = measure_chunk_dedup(baseline_seqs, CHUNK_SIZE)
print_chunk("xgboost large (baseline)", chunk_stats, CHUNK_SIZE, max_tensors=10)
print(f"\n  Measured in {time.time() - t0:.1f}s")

### Skeleton size over time

The `__skeleton__` blob contains all non-tree XGBoost metadata. Its growth rate
determines how much overhead tensorcas pays per step even with no new tensors.

In [ ]:
print("Skeleton size over time:")
print(f"  {'Step':>6} {'Rounds':>8} {'Skeleton bytes':>16} {'Growth':>8}")
print(f"  {'-'*6} {'-'*8} {'-'*16} {'-'*8}")

prev_size = None
for i, seqs in enumerate(baseline_seqs):
    skeleton = seqs["__skeleton__"]
    size = skeleton.nbytes
    growth = f"+{_fmt_bytes(size - prev_size)}" if prev_size is not None else "—"
    rounds = (i + 1) * ROUNDS_PER_STEP
    print(f"  {i+1:>6} {rounds:>8} {_fmt_bytes(size):>16} {growth:>8}")
    prev_size = size

## Crossover analysis: at what scale does tensorcas beat DVC?

For small XGBoost models the binary UBJ format compresses well, giving DVC
an advantage. This section sweeps model sizes to find the crossover point.

In [ ]:
import tempfile
from tensorcas.store import tensorcasStore
from tensorcas.adapters.xgboost import XGBoostAdapter

CROSSOVER_STEPS = [
    (5,  10),   #  50 rounds total
    (10, 10),   # 100 rounds
    (5,  50),   # 250 rounds
    (10, 50),   # 500 rounds
    (20, 50),   # 1 000 rounds (baseline)
]

print(f"\n{'Total rounds':>12} {'DVC bytes':>12} {'tensorcas bytes':>12} {'Ratio':>8} {'Winner':>8}")
print("=" * 60)

xgb_adapter = XGBoostAdapter()

for n_steps, rounds_per_step in CROSSOVER_STEPS:
    boosters = train_sequence(seed=0, n_steps=n_steps, rounds_per_step=rounds_per_step)

    # Write checkpoints for DVC measurement
    with tempfile.TemporaryDirectory() as tmpdir:
        ckpt_dir = Path(tmpdir) / "ckpts"
        ckpt_dir.mkdir()
        for step, b in enumerate(boosters, 1):
            b.save_model(str(ckpt_dir / f"step_{step:04d}.ubj"))
        dvc = dvc_bytes(sorted(ckpt_dir.glob("*.ubj")))

        # tensorcas measurement
        tensorcas_root = Path(tmpdir) / "tensorcas"
        with tensorcasStore(root=tensorcas_root, run_id="bench", adapter=xgb_adapter) as store:
            for step, b in enumerate(boosters, 1):
                store.save(b, step=step)
        tensorcas = tensorcas_bytes(tensorcas_root)

    total_rounds = n_steps * rounds_per_step
    ratio = tensorcas / dvc if dvc else 0.0
    winner = "tensorcas" if tensorcas < dvc else "DVC"
    print(f"  {total_rounds:>10} {_fmt_bytes(dvc):>12} {_fmt_bytes(tensorcas):>12} {ratio:>8.3f} {winner:>8}")

## Cross-run chunk sharing

Two independent runs with different seeds but the same hyperparameters.

In [ ]:
t0 = time.time()
print("Training run 2 (seed=99)...", end=" ", flush=True)
run2_boosters = train_sequence(seed=99)
print(f"{time.time() - t0:.1f}s")

In [ ]:
run2_seqs = [extract_xgboost(b) for b in run2_boosters]
crossrun_stats = measure_crossrun(baseline_seqs, run2_seqs, CHUNK_SIZE)
print_crossrun("xgboost large (seed=0 vs seed=99)", crossrun_stats)

## DVC vs tensorcas — full 1 000-round comparison

In [ ]:
xgboost_dir = CHECKPOINT_DIR / "xgboost"
xgboost_dir.mkdir(parents=True, exist_ok=True)

for step, booster in enumerate(baseline_boosters, 1):
    booster.save_model(str(xgboost_dir / f"step_{step * ROUNDS_PER_STEP:06d}.ubj"))

files = sorted(xgboost_dir.glob("*.ubj"))
total_raw = sum(p.stat().st_size for p in files)
print(f"{len(files)} checkpoints written ({_fmt_bytes(total_raw)} total)")

In [ ]:
from tensorcas.adapters.xgboost import XGBoostAdapter

print("Running DVC vs tensorcas comparison...", end=" ", flush=True)
t0 = time.time()
result = run_dvc_comparison(
    "xgboost", baseline_boosters, XGBoostAdapter(), xgboost_dir
)
print(f"{time.time() - t0:.1f}s")

print_dvc_comparison([result])
print()
print("Note: XGBoost binary UBJ format compresses well, giving DVC an advantage")
print("at small model sizes. tensorcas's advantage emerges at larger round counts where")
print("tree reuse dominates over skeleton overhead.")

## Summary

| Metric | Result |
|--------|--------|
| No-op rate (individual trees) | Expected ~100% for existing trees |
| Skeleton overhead | Grows with round count; unavoidable |
| DVC crossover | tensorcas wins once tree reuse > skeleton overhead |
| Cross-run sharing | Low — different seeds → different tree splits |

## Scaling benchmark: save latency vs round count

Sweeps total round counts from 500 to 5 000 (fixed 50 rounds/step).

Unlike sklearn (where all tensors are unchanged after writing), XGBoost always
changes `__skeleton__` every step — one CAS write per step regardless of round
count. Individual tree tensors are stable once written.

Measures the last 5 steps of each run (pure no-op path for trees, one changed
tensor for `__skeleton__`). Should be flat — cost per step is O(1) changed
tensors, not O(total trees).

In [ ]:
import statistics
import tempfile
from tensorcas.store import tensorcasStore
from tensorcas.adapters.xgboost import XGBoostAdapter

# (total_rounds, rounds_per_step) — step count varies, step size fixed at 50
SCALING_CONFIGS = [
    ( 500, 50),   #  10 steps
    (1000, 50),   #  20 steps  ← baseline
    (2000, 50),   #  40 steps
    (3000, 50),   #  60 steps
    (5000, 50),   # 100 steps
]

def _measure_save_latency(boosters, n_sample: int = 5) -> dict:
    """
    Save all boosters in order through a single tensorcasStore.
    Time only the last n_sample steps (warm no-op path for trees;
    __skeleton__ still changes every step).
    """
    with tempfile.TemporaryDirectory() as tmp:
        with tensorcasStore(root=Path(tmp), run_id="scale", adapter=XGBoostAdapter()) as store:
            for step, b in enumerate(boosters[:-n_sample], 1):
                store.save(b, step=step)
            latencies = []
            for i, b in enumerate(boosters[-n_sample:], len(boosters) - n_sample + 1):
                t0 = time.perf_counter()
                store.save(b, step=i)
                latencies.append((time.perf_counter() - t0) * 1000)
    return {
        "median_ms": statistics.median(latencies),
        "min_ms": min(latencies),
        "max_ms": max(latencies),
    }


print(f"{'Total rounds':>13} {'Steps':>6} {'Tensors/ckpt':>14} {'Median save':>13} {'Min':>8} {'Max':>8}  {'Scaling'}")
print("=" * 82)

prev_median = None
for total_rounds, rounds_per_step in SCALING_CONFIGS:
    n_steps = total_rounds // rounds_per_step
    t0 = time.time()
    print(f"  Training {total_rounds} rounds ({n_steps} steps)...", end=" ", flush=True)
    boosters = train_sequence(seed=0, n_steps=n_steps, rounds_per_step=rounds_per_step)
    print(f"{time.time() - t0:.1f}s", end=" | measuring... ", flush=True)

    # tensors/ckpt = 1 skeleton + 1 per tree
    tensors_per_ckpt = len(extract_xgboost(boosters[-1]))
    lat = _measure_save_latency(boosters)
    print("done")

    flag = ""
    if prev_median is not None and lat["median_ms"] > prev_median * 2:
        flag = "  ← superlinear"
    print(
        f"  {total_rounds:>13,} {n_steps:>6} {tensors_per_ckpt:>14,} "
        f"  {lat['median_ms']:>9.1f}ms {lat['min_ms']:>6.1f}ms {lat['max_ms']:>6.1f}ms{flag}"
    )
    prev_median = lat["median_ms"]

### Results (Colab, March 2026)

```
Total rounds  Steps   Tensors/ckpt   Median save      Min      Max  Scaling
==================================================================================
         500     10            501        50.4ms   46.1ms   54.3ms
       1,000     20          1,001        60.9ms   59.5ms   66.7ms
       2,000     40          2,001       103.4ms   95.6ms  301.5ms
       3,000     60          3,001       133.4ms  131.6ms  210.7ms
       5,000    100          5,001       208.7ms  205.7ms  539.0ms
```

Save latency grows roughly linearly with round count (~4× from 500→5 000 rounds).
This is expected and structural: `__skeleton__` grows ~400B per 50 rounds and is
written fresh every step — at 5 000 rounds it is ~40KB per CAS write.
Individual tree tensors are fully cached after the first write (100% no-op).

The remaining cost is O(`__skeleton__` size), not O(total trees).